In [ ]:
from libraries import *
from parameters import *
from sklearn.cross_decomposition import PLSCanonical, PLSRegression, CCA
from sklearn.model_selection import cross_val_score
from numpy import asarray
from numpy import savetxt
from adjustText import adjust_text
import seaborn as sns
from sklearn.cluster import SpectralBiclustering
%matplotlib inline

In [ ]:
%load_ext rpy2.ipython

In [ ]:
guideModules = pd.read_csv(reference_path("ME_GuideModules_leiden_6_Modules.csv"), index_col=0)

In [ ]:
guideModules.loc[["Rack1", "Rnf128", "Socs3"],:]

In [ ]:
coefsAll = pd.read_csv(textfiles_path("ME_SignificantBetaCoefs.csv"),
                       header=0, index_col=0)
coefsAll.index = [cond.split('_')[1] for cond in coefsAll.index]

In [ ]:
coefsAllRemoved = coefsAll.loc[[x not in ["Rack1", "Rnf128", "Socs3"] for x in coefsAll.index],:]
coefsAllRemoved = coefsAllRemoved.loc[:,[x not in ["Rack1", "Rnf128", "Socs3"] for x in coefsAll.columns]]

In [ ]:
koGuidesAnnDat = sc.AnnData(X=coefsAll)
sc.pp.pca(koGuidesAnnDat, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(koGuidesAnnDat)
connectivitiesOriginal = pd.DataFrame(koGuidesAnnDat.obsp['connectivities'].A)
connectivitiesOriginal.index = coefsAll.index
connectivitiesOriginal.columns = coefsAll.index

In [ ]:
coefsAllRemoved.shape

In [ ]:
koGuidesAnnDat_rem = sc.AnnData(X=coefsAllRemoved)
sc.pp.pca(koGuidesAnnDat_rem, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(koGuidesAnnDat_rem)
connectivitiesRem = pd.DataFrame(koGuidesAnnDat_rem.obsp['connectivities'].A)
connectivitiesRem.index = coefsAllRemoved.index
connectivitiesRem.columns = coefsAllRemoved.index

In [ ]:
connectivitiesOriginal = connectivitiesOriginal.loc[connectivitiesRem.index, connectivitiesRem.columns]

In [ ]:
guideModules = guideModules.loc[[x in connectivitiesOriginal.index for x in guideModules.index],:]

In [ ]:
connectivitiesOriginal = connectivitiesOriginal.loc[guideModules.index, guideModules.index]

In [ ]:
guideModules

In [ ]:
%%R -i connectivitiesOriginal,guideModules -w 9 -h 8 -u in

library("reshape")
connectivitiesOriginal$KOGene = rownames(connectivitiesOriginal)
connectivitiesOriginalMelted = melt(connectivitiesOriginal, id.var="KOGene")

connectivitiesOriginalMelted$KOGene = factor(connectivitiesOriginalMelted$KOGene, 
                                             levels=guideModules$GuideName)

connectivitiesOriginalMelted = connectivitiesOriginalMelted[order(connectivitiesOriginalMelted$KOGene,
                                                                  -connectivitiesOriginalMelted$value),]

colnames(connectivitiesOriginalMelted) = c("GuideName", "Neighbour","Connectivity")

connectivitiesOriginalMelted = connectivitiesOriginalMelted[connectivitiesOriginalMelted$Connectivity != 0,]
connectivitiesOriginalMelted$KOGroup1 = guideModules[connectivitiesOriginalMelted$GuideName,
                                                     "NewGuideGroup"]
connectivitiesOriginalMelted$KOGroup2 = guideModules[connectivitiesOriginalMelted$Neighbour,
                                                     "NewGuideGroup"]



write.csv(connectivitiesOriginalMelted, "outputs/ConnectivitiesOriginal.csv", row.names=FALSE)


library(pheatmap)

connectivitiesOriginal$KOGene = NULL


annoCols<-list(NewGuideGroup=c(M2="#1f77b4",
                             M3="#ff7f0e",
                             M6="#279e68",
                             M5="#d62728", 
                             M1="#aa40fc", 
                             M4="#8c564b"))

guideModules$GuideName = NULL
guideModules$GuideGroup = NULL
guideModules$GuideColor = NULL

connectivitiesOriginal[connectivitiesOriginal > 0.1]  = 0.1
connectivitiesOriginal[connectivitiesOriginal < -0.1]  = -0.1

pheatmap(connectivitiesOriginal,
         cluster_cols = FALSE,
         cluster_rows = FALSE,
         fontsize_row = 3,
         fontsize_col = 3,
         annotation_row=guideModules,
         annotation_col=guideModules,
         annotation_colors = annoCols,
         clustering_method="ward.D",
         clustering_distance_rows="euclidean", colorRampPalette(c("white", "red"))(200))


In [ ]:
connectivitiesRem = connectivitiesRem.loc[guideModules.index, guideModules.index]

In [ ]:
connectivitiesRem.shape

In [ ]:
%%R -i connectivitiesRem,guideModules -w 9 -h 8 -u in


connectivitiesRem$KOGene = rownames(connectivitiesRem)
connectivitiesRemMelted = melt(connectivitiesRem, id.var="KOGene")

connectivitiesRemMelted$KOGene = factor(connectivitiesRemMelted$KOGene, 
                                        levels=guideModules$GuideName)

connectivitiesRemMelted = connectivitiesRemMelted[order(connectivitiesRemMelted$KOGene,
                                                        -connectivitiesRemMelted$value),]

colnames(connectivitiesRemMelted) = c("GuideName", "Neighbour","Connectivity")

connectivitiesRemMelted = connectivitiesRemMelted[connectivitiesRemMelted$Connectivity != 0,]

connectivitiesRemMelted$KOGroup1 = guideModules[connectivitiesRemMelted$GuideName,
                                                     "NewGuideGroup"]
connectivitiesRemMelted$KOGroup2 = guideModules[connectivitiesRemMelted$Neighbour,
                                                     "NewGuideGroup"]

write.csv(connectivitiesRemMelted, "outputs/ConnectivitiesUpdated.csv", row.names=FALSE)




library(pheatmap)

connectivitiesRem$KOGene  = NULL

annoCols<-list(NewGuideGroup=c(M2="#1f77b4",
                             M3="#ff7f0e",
                             M6="#279e68",
                             M5="#d62728", 
                             M1="#aa40fc", 
                             M4="#8c564b"))

guideModules$GuideName = NULL
guideModules$GuideGroup = NULL
guideModules$GuideColor = NULL

connectivitiesRem[connectivitiesRem > 0.1]  = 0.1
connectivitiesRem[connectivitiesRem < -0.1]  = -0.1

pheatmap(connectivitiesRem,
         cluster_cols = FALSE,
         cluster_rows = FALSE,
         fontsize_row = 3,
         fontsize_col = 3,
         annotation_row=guideModules,
         annotation_col=guideModules,
         annotation_colors = annoCols,
         clustering_method="ward.D",
         clustering_distance_rows="euclidean", colorRampPalette(c("white", "red"))(200))


In [ ]:
%%R

lappend <- function(lst, obj) {
  lst[[length(lst)+1]] <- obj
  return(lst)
}
allKOGenes = list()
for( elem in unique(connectivitiesOriginalMelted$GuideName)){
    
    originalNeighbors = connectivitiesOriginalMelted[connectivitiesOriginalMelted$GuideName == elem,
                                                     "Neighbour"][1:15]
    
    newNeighbors = connectivitiesRemMelted[connectivitiesRemMelted$GuideName == elem,
                                                     "Neighbour"][1:15]
    
    allKOGenes = lappend(allKOGenes, c("KOGene" = elem,
                                       "NumOverlappingNeighbors" = length(originalNeighbors[originalNeighbors %in% newNeighbors]),
                                       "KOModule"=unique(connectivitiesOriginalMelted[connectivitiesOriginalMelted$GuideName == elem, "KOGroup1"])))
    
    
}

allKOGenes <- data.frame(do.call(rbind,allKOGenes))
write.csv(allKOGenes, "outputs/NumberOfOverlappingNeighbours.csv", row.names=FALSE)

In [ ]:
np.random.seed(1)

In [ ]:
sc.tl.leiden(koGuidesAnnDat, resolution=0.64, random_state=0)

In [ ]:
sc.tl.leiden(koGuidesAnnDat_rem, resolution=0.67, random_state=0)

In [ ]:
koGuidesAnnDat.obs["red_leiden"] = koGuidesAnnDat_rem.obs["leiden"]

In [ ]:
sum(koGuidesAnnDat.obs["leiden"] == koGuidesAnnDat.obs["red_leiden"])

In [ ]:
koGuidesAnnDat.obs.loc[koGuidesAnnDat.obs["leiden"] != koGuidesAnnDat.obs["red_leiden"],:]

In [ ]:
koGuidesAnnDat_rem.obs["leiden"].value_counts()